# 01 · Data understanding
**Project:** Kenya Retail Intelligence. **Company:** *Savanna Fresh Supermarkets* (fictional).

> All transaction-level data in this project is **synthetic**, produced by `src/generate_data.py` for a fictional chain.
> It does not describe any real retailer. Public datasets were reviewed first (see `reports/methodology.md`); none were
> Kenyan, store-based and complete with cost, inventory and promotion data.

This notebook answers: *what tables do we have, what does one row mean, what period is covered, and how do the tables relate?*

In [1]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # project root, so `src` is importable
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.data_loader import get_engine, query, PROJECT_ROOT
from src import analysis as an
an.set_style()
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)
engine = get_engine()

## 1. Row counts and grain
The grain (what one row represents) decides how every later aggregation must be written.

In [2]:
grain = {
 "branches": "one store", "products": "one SKU (product x brand x pack size)", "customers": "one loyalty member",
 "promotions": "one chain-wide price promotion on one product", "transactions": "one till receipt (basket)",
 "transaction_items": "one product line on a receipt", "returns": "one returned product line",
 "inventory_snapshots": "Sunday closing stock for one product in one branch"}
counts = query(engine, """
SELECT 'branches' t, COUNT(*) n FROM retail.branches UNION ALL SELECT 'products', COUNT(*) FROM retail.products
UNION ALL SELECT 'customers', COUNT(*) FROM retail.customers UNION ALL SELECT 'promotions', COUNT(*) FROM retail.promotions
UNION ALL SELECT 'transactions', COUNT(*) FROM retail.transactions UNION ALL SELECT 'transaction_items', COUNT(*) FROM retail.transaction_items
UNION ALL SELECT 'returns', COUNT(*) FROM retail.returns UNION ALL SELECT 'inventory_snapshots', COUNT(*) FROM retail.inventory_snapshots""")
counts["grain"] = counts["t"].map(grain)
counts

,t,n,grain
0,branches,12,one store
1,promotions,110,one chain-wide price promotion on one product
2,products,203,one SKU (product x brand x pack size)
3,returns,2996,one returned product line
4,customers,9000,one loyalty member
5,inventory_snapshots,249487,Sunday closing stock for one product in one br...
6,transactions,289937,one till receipt (basket)
7,transaction_items,1198181,one product line on a receipt


## 2. Column types (clean `retail` schema)

In [3]:
cols = query(engine, """SELECT table_name, column_name, data_type, is_nullable
FROM information_schema.columns WHERE table_schema = 'retail' AND table_name NOT LIKE 'v_%%'
ORDER BY table_name, ordinal_position""")
cols.groupby("table_name").apply(lambda g: ", ".join(f"{c} ({t})" for c, t in zip(g.column_name, g.data_type))).to_frame("columns")

,columns
table_name,
branches,"branch_id (character varying), branch_name (ch..."
customers,"customer_id (character varying), customer_segm..."
dq_exceptions,"exception_id (bigint), source_table (character..."
inventory_snapshots,"inventory_id (bigint), product_id (character v..."
products,"product_id (character varying), product_name (..."
promotions,"promotion_id (character varying), product_id (..."
returns,"return_line_id (bigint), transaction_id (chara..."
transaction_items,"transaction_item_id (bigint), transaction_id (..."
transactions,"transaction_id (character varying), transactio..."


## 3. Master data

In [4]:
query(engine, 'SELECT * FROM retail.branches ORDER BY branch_id')

,branch_id,branch_name,county,town,branch_type,opening_date
0,BR01,Savanna Fresh Westlands,Nairobi,Westlands,Hypermarket,2018-03-01
1,BR02,Savanna Fresh Embakasi,Nairobi,Embakasi,Supermarket,2019-07-15
2,BR03,Savanna Fresh Kasarani,Nairobi,Kasarani,Express,2021-02-01
3,BR04,Savanna Fresh Nyali,Mombasa,Nyali,Hypermarket,2019-11-01
4,BR05,Savanna Fresh Mombasa CBD,Mombasa,Mombasa,Supermarket,2020-04-01
5,BR06,Savanna Fresh Kisumu,Kisumu,Kisumu,Supermarket,2020-09-01
6,BR07,Savanna Fresh Nakuru,Nakuru,Nakuru,Hypermarket,2019-05-01
7,BR08,Savanna Fresh Naivasha,Nakuru,Naivasha,Express,2022-03-01
8,BR09,Savanna Fresh Thika,Kiambu,Thika,Supermarket,2021-08-01
9,BR10,Savanna Fresh Ruiru,Kiambu,Ruiru,Express,2025-06-02


In [5]:
prod = query(engine, 'SELECT category, COUNT(*) AS skus, COUNT(DISTINCT brand) AS brands, ROUND(AVG(unit_cost),0) AS avg_unit_cost FROM retail.products GROUP BY category ORDER BY skus DESC')
prod

,category,skus,brands,avg_unit_cost
0,Staples,36,5,262.00
1,Beverages,24,4,149.00
2,Household & Cleaning,24,4,177.00
3,Fresh Produce,24,4,79.00
4,Snacks & Confectionery,20,4,82.00
5,Personal Care,18,4,156.00
6,Dairy,18,4,209.00
7,Bakery,15,3,69.00
8,Baby Care,12,3,468.00
9,Meat & Poultry,12,3,481.00


In [6]:
query(engine, 'SELECT customer_segment, COUNT(*) AS customers FROM retail.customers GROUP BY 1 ORDER BY 2 DESC')

,customer_segment,customers
0,Household,5864
1,Young Professional,1779
2,Small Business,722
3,Senior,635


## 4. Period coverage
Two complete years, November 2024 to October 2026. Ruiru (BR10) opened on 2 June 2025, so it has no full comparison year, and like-for-like comparisons exclude it.

In [7]:
query(engine, """SELECT branch_id, MIN(transaction_ts)::date AS first_sale, MAX(transaction_ts)::date AS last_sale,
 COUNT(*) AS transactions FROM retail.transactions GROUP BY 1 ORDER BY 1""")

,branch_id,first_sale,last_sale,transactions
0,BR01,2024-11-01,2026-10-31,37904
1,BR02,2024-11-01,2026-10-31,27120
2,BR03,2024-11-01,2026-10-31,18739
3,BR04,2024-11-01,2026-10-31,34474
4,BR05,2024-11-01,2026-10-31,23441
5,BR06,2024-11-01,2026-10-31,24215
6,BR07,2024-11-01,2026-10-31,30729
7,BR08,2024-11-01,2026-10-31,16377
8,BR09,2024-11-01,2026-10-31,23042
9,BR10,2025-06-02,2026-10-31,13332


## 5. A sample basket, end to end
Following one receipt through the joins is the quickest way to check that the relationships behave as expected.

In [8]:
tid = query(engine, "SELECT transaction_id FROM retail.transactions WHERE customer_id IS NOT NULL ORDER BY transaction_id LIMIT 1 OFFSET 5000").iloc[0, 0]
query(engine, """SELECT t.transaction_id, t.transaction_ts, b.branch_name, c.customer_segment, t.payment_method,
  p.product_name, ti.quantity, ti.unit_price, ti.discount_amount, ti.promotion_id
FROM retail.transactions t JOIN retail.branches b USING (branch_id)
LEFT JOIN retail.customers c USING (customer_id)
JOIN retail.transaction_items ti USING (transaction_id) JOIN retail.products p USING (product_id)
WHERE t.transaction_id = :tid""", tid=tid)

,transaction_id,transaction_ts,branch_name,customer_segment,payment_method,product_name,quantity,unit_price,discount_amount,promotion_id
0,T0007938,2024-11-24 13:33:13,Savanna Fresh Thika,Young Professional,Mobile Money,Savanna Value Maize Flour 2kg,2,160.00,0.00,None
1,T0007938,2024-11-24 13:33:13,Savanna Fresh Thika,Young Professional,Mobile Money,Savanna Farm Tomatoes 1kg,1,118.00,0.00,None
2,T0007938,2024-11-24 13:33:13,Savanna Fresh Thika,Young Professional,Mobile Money,Savanna Value Yoghurt 500ml,1,144.00,30.63,None
3,T0007938,2024-11-24 13:33:13,Savanna Fresh Thika,Young Professional,Mobile Money,Tropika Tea Leaves 250g,1,155.00,0.00,None


## 6. Relationships
```
branches 1──* transactions *──0..1 customers
transactions 1──* transaction_items *──1 products
transaction_items *──0..1 promotions *──1 products
branches 1──* inventory_snapshots *──1 products
```
`customer_id` is nullable because about 38% of baskets are not scanned with a loyalty card. Customer-level analysis therefore covers only identified baskets, while revenue analysis covers all of them.

**Takeaways for the next steps:** analysis must aggregate lines before counting baskets (fan-out risk), compare branches like-for-like because of BR10, and treat customer metrics as a loyalty-member subset.